## Imports

In [24]:
import warnings
warnings.filterwarnings("ignore")

In [25]:
import math
import os
import time
import requests
import pandas as pd
import json
import numpy as np
from pathlib import Path
from typing import Optional, Dict, List, Union
from unidecode import unidecode
import matplotlib.pyplot as plt

import pandas_gbq
from google.auth import default
from google.cloud import bigquery
from google.api_core.exceptions import NotFound

In [26]:
from funcoes_escoragem import *

## Diretório

In [27]:
BASE_DIR = Path("data")
RAW_DIR = BASE_DIR / "raw"
TRUSTED_DIR = BASE_DIR / "trusted"
ANALYTICS_DIR = BASE_DIR / "analytics"

for path in [RAW_DIR, TRUSTED_DIR, ANALYTICS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# Funil Modelos

In [28]:
project_id = 'loft-dl-fintech'

In [29]:
query = '''
WITH tb_leads AS (
  SELECT 
    rf.contract_id,
    date(rf.dt_lead) as dt_lead,
    date(cf.requested_at) as requested_at,
    date(rf.dt_proposta_iniciada) as iniciada_at,
    date(rf.dt_proposta_enviada) as enviada_at,
    date(rf.activated_at) as activated_at,
    date(rf.cancelled_at) as cancelled_at,
    date(rf.dt_saida) as dt_saida,
    cf.tipo_contrato,
    rd.product_nm,
    cf.modeloBlend,
    cf.bureau_nm,
    cf.modelo_blend,
    case when cf.modeloBlend in ('BLEND_4', 'BVS_CUSTOM_V2', 'HFT1') then 'BLEND4'
         when cf.bureau_nm in ('BLEND_4', 'BVS_CUSTOM_V2', 'HFT1') then 'BLEND4'
         when cf.modeloBlend in ('BLEND3_3', 'BVS_CUSTOM', 'HVA4') then 'BLEND3'
         when cf.bureau_nm in ('BLEND3_3', 'BVS_CUSTOM', 'HVA4') then 'BLEND3'
         else modeloBlend end as bureau_nm_ajust,
    case when cf.modeloBlend in ('BVS_CUSTOM_V2', 'HFT1', 'BVS_CUSTOM', 'HVA4') then 1
         when cf.bureau_nm in ('BVS_CUSTOM_V2', 'HFT1', 'BVS_CUSTOM', 'HVA4') then 1 else 0 end as is_fallback,
    cast(rf.total_rental_value_informed_nr as FLOAT64) as total_rental_value_informed_nr,
    cast(rf.rental_value_nr as FLOAT64) as rental_value_nr,
    cf.agency_id,
    cf.contract_city_nm,
    cf.person_age,
    cf.qtd_proponentes,
    cf.score_imobiliaria,
    cf.person_restriction_total_value,
    ca.bvs_cust_score_nr,
    ca.blend_regressao_predict_nr,
    cf.rating_score_ds,
    cf.person_deemed_income,
    rd.pre_analysis_result,
    rd.lead_elegivel,
    rd.proposta_iniciada,
    rd.proposta_enviada,
    rd.proposta_aprovada,
    rd.proposta_ativada,
    rd.is_activeted
  FROM loft-dl-fintech.cp_gold.requests_fact AS rf
  LEFT JOIN loft-dl-fintech.cp_gold.requests_dim AS rd
    ON rf.contract_id = rd.contract_id
  LEFT JOIN loft-dl-fintech.cp_gold.credit_fact AS cf
    ON rf.contract_id = cf.contract_id
  LEFT JOIN loft-dl-fintech.cp_silver.int_credit_analyses AS ca
    ON rf.contract_id = ca.contract_id
  WHERE cf.tipo_contrato = 'PF'
),
first_defaults AS (
  SELECT
    contract_id
    , DATE(MIN(pendency_created_at)) AS first_comunicacao_date
    , DATE(MIN(pendency_at)) AS first_competencia_date
  FROM loft-dl-fintech.cp_gold.watchlist_fact
  WHERE pendency_type = "Inadimplência"
  GROUP BY contract_id
)
SELECT
  l.*
  , fd.first_comunicacao_date
  , fd.first_competencia_date
FROM tb_leads AS l
LEFT JOIN first_defaults as fd
  on l.contract_id = fd.contract_id
  and l.activated_at is not null
where 1=1
    and date(l.requested_at) >= date('2026-07-01')
    and date(l.requested_at) < date(current_date())
'''

In [30]:
df_funil = pandas_gbq.read_gbq(query, project_id=project_id)
df_funil

Downloading: 100%|██████████|


,contract_id,dt_lead,requested_at,iniciada_at,enviada_at,activated_at,cancelled_at,dt_saida,tipo_contrato,product_nm,...,person_deemed_income,pre_analysis_result,lead_elegivel,proposta_iniciada,proposta_enviada,proposta_aprovada,proposta_ativada,is_activeted,first_comunicacao_date,first_competencia_date
0,4899052,2026-11-04,2026-11-04,NaT,NaT,NaT,2026-11-04,NaT,PF,None,...,1918.000000000,REPROVAR,False,False,False,False,False,False,NaT,NaT
1,4896596,2026-11-04,2026-11-04,NaT,NaT,NaT,2026-11-04,NaT,PF,None,...,1712.500000000,REPROVAR,False,False,False,False,False,False,NaT,NaT
2,4898662,2026-11-04,2026-11-04,NaT,NaT,NaT,2026-11-04,NaT,PF,None,...,0E-9,REPROVAR,False,False,False,False,False,False,NaT,NaT
3,4897562,2026-11-04,2026-11-04,NaT,NaT,NaT,2026-11-04,NaT,PF,None,...,0E-9,REPROVAR,False,False,False,False,False,False,NaT,NaT
4,4900529,2026-11-04,2026-11-04,NaT,NaT,NaT,2026-11-04,NaT,PF,None,...,0E-9,REPROVAR,False,False,False,False,False,False,NaT,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
566235,4442254,2026-07-30,2026-07-30,2026-07-30,2026-07-30,2026-07-30,NaT,NaT,PF,Fit,...,3425.000000000,APROVAR,True,True,True,True,True,True,2026-10-29,2026-09-10
566236,4444192,2026-07-30,2026-07-30,2026-07-30,2026-07-30,2026-07-31,NaT,NaT,PF,Fit,...,12741.000000000,APROVAR,True,True,True,True,True,True,NaT,NaT
566237,4445452,2026-07-30,2026-07-30,2026-07-30,2026-07-30,NaT,NaT,NaT,PF,Smart,...,30003.000000000,APROVAR,True,True,True,True,False,False,NaT,NaT
566238,4441514,2026-07-30,2026-07-30,2026-08-01,2026-08-01,NaT,NaT,NaT,PF,Up,...,2534.500000000,APROVAR,True,True,True,True,False,False,NaT,NaT


In [31]:
df_funil.groupby("bureau_nm_ajust", dropna=False).size()

bureau_nm_ajust
BLEND3                  111259
BLEND3_4                116720
BLEND4                  336750
BLEND_REGRESSAO_2026      1278
NaN                        233
dtype: int64

In [32]:
df_funil.groupby("bureau_nm", dropna=False).size()

bureau_nm
                            72
BLEND3_3                104775
BLEND3_4                112954
BLEND_4                 316185
BLEND_REGRESSAO_2026      1188
BVS_CUSTOM                 481
BVS_CUSTOM_V2             5386
HFT1                      1172
HVA3                     21980
HVA4                      1814
NaN                        233
dtype: int64

In [33]:
df_funil.groupby("bureau_nm_ajust", dropna=False).size()

bureau_nm_ajust
BLEND3                  111259
BLEND3_4                116720
BLEND4                  336750
BLEND_REGRESSAO_2026      1278
NaN                        233
dtype: int64

In [34]:
df_funil.groupby("modeloBlend", dropna=False).size()

modeloBlend
BLEND3_3                110250
BLEND3_4                117522
BLEND_4                 336453
BLEND_REGRESSAO_2026      1485
BVS_CUSTOM_V2              165
HFT1                       132
NaN                        233
dtype: int64

In [35]:
bvs = pd.to_numeric(df_funil["bvs_cust_score_nr"], errors="coerce")
score = pd.to_numeric(df_funil["blend_regressao_predict_nr"], errors="coerce")

conditions = [
    bvs <= 334,                         # corte customizado BVS → E
    score.between(763, 1000),           # 763 – 1000
    score.between(704, 762),            # 704 – 762
    score.between(653, 703),            # 653 – 703
    score.between(607, 652),            # 607 – 652
    score.between(562, 606),            # 562 – 606
    score.between(520, 561),            # 520 – 561
    score.between(480, 519),            # 480 – 519
    score.between(443, 479),            # 443 – 479
    score.between(408, 442),            # 408 – 442
    score.between(375, 407),            # 375 – 407
    score.between(343, 374),            # 343 – 374
    score.between(307, 342),            # 307 – 342
    score.between(0, 306),              # 0 – 306
]

choices = [
    "E.BVS",      # override BVS ≤ 334
    "1.A+",
    "2.A",
    "3.A",
    "4.B+",
    "5.B+",
    "6.B",
    "7.B",
    "8.C",
    "9.D+",
    "10.D",
    "11.D",
    "E-1.E",
    "E-2.E",
]

df_funil["rating_cl_pol_blend4"] = np.select(conditions, choices, default=None)

In [36]:
bvs = pd.to_numeric(df_funil["bvs_cust_score_nr"], errors="coerce")
score = pd.to_numeric(df_funil["blend_regressao_predict_nr"], errors="coerce")

conditions = [
    bvs <= 334,                         # corte customizado BVS → E
    score.between(763, 1000),           # 763 – 1000
    score.between(704, 762),            # 704 – 762
    score.between(653, 703),            # 653 – 703
    score.between(607, 652),            # 607 – 652
    score.between(562, 606),            # 562 – 606
    score.between(520, 561),            # 520 – 561
    score.between(480, 519),            # 480 – 519
    score.between(443, 479),            # 443 – 479
    score.between(408, 442),            # 408 – 442
    score.between(375, 407),            # 375 – 407
    score.between(343, 374),            # 343 – 374
    score.between(307, 342),            # 307 – 342
    score.between(0, 306),              # 0 – 306
]

choices = [
    "9.E-BVS",      # override BVS ≤ 334
    "1.A+",
    "2.A",
    "2.A",
    "3.B+",
    "3.B+",
    "4.B",
    "4.B",
    "5.C",
    "6.D+",
    "7.D",
    "7.D",
    "8.E",
    "8.E",
]

df_funil["rating_pol_blend4"] = np.select(conditions, choices, default=None)
df_funil["rating_pol_blend4"] = df_funil["rating_pol_blend4"].fillna("N/I")

In [37]:
df_funil.groupby("rating_pol_blend4", dropna=False).size()

rating_pol_blend4
1.A+        19103
2.A         50310
3.B+        55312
4.B         58010
5.C         29589
6.D+        30353
7.D         61806
8.E        152827
9.E-BVS     10295
N/I         98635
dtype: int64

In [38]:
df_funil.to_csv(ANALYTICS_DIR/"df_funil_blend4.csv", index=False)

In [39]:
df_funil[df_funil["contract_id"]==4358668][["contract_id", "modeloBlend", "bureau_nm", "modelo_blend", "bureau_nm_ajust", "is_fallback"]]

,contract_id,modeloBlend,bureau_nm,modelo_blend,bureau_nm_ajust,is_fallback
160080,4358668,BLEND_4,BLEND3_3,BLEND_4,BLEND4,0


In [40]:
df_blend4_dev = pd.read_csv(ANALYTICS_DIR/"df_predict_blend4.csv")

In [41]:
df_blend4_dev.columns.tolist()

['CPF_CNPJ',
 'contract_id',
 'requested_at',
 'data_ultima_consulta',
 'qtd_proponentes',
 'codigo_cidade_ibge',
 'score_imobiliaria',
 'model',
 'id_consulta',
 'id_funcionalidade',
 'documento',
 'rating',
 'result',
 'message_rendaConsiderada',
 'income',
 'message_rendaUtilizada',
 'message_fatorCorrecaoRenda',
 'message_limiteCalculadoContrato',
 'qtde_restricoes__consulta_realizada',
 'message_restricoesPorModalidadeContrato',
 'message_regra_estouro_limite',
 'message_valorRestricaoTotal',
 'message_regua',
 'message_score',
 'message_blendRegressaoPredict',
 'message_decisao',
 'message_classificacao',
 'message_existeErroSerasa',
 'message_existeErroPowerCurve',
 'message_existeErroBvsCustom',
 'message_motor',
 'message_regras_situacao_cadastral_serasa',
 'score_proposto__bvs',
 'SERASA_CHSV5',
 'message_ratings_BLEND_4',
 'message_feature_flags_analiseAutomaticaDeCredito',
 'message_feature_flags_modeloBlend',
 'flag_tem__contratos_anteriores',
 'flag_teve_boleto_atrasado__

In [42]:
df_blend4_dev[df_blend4_dev["contract_id"]==4358668][["contract_id", "model", "message_decisao", "message_scores_HVA4", "message_ratings_BVS_CUSTOM", "pessoa_scores_HVA4"]]

,contract_id,model,message_decisao,message_scores_HVA4,message_ratings_BVS_CUSTOM,pessoa_scores_HVA4


In [43]:
df_funil[df_funil["is_fallback"]==1]

,contract_id,dt_lead,requested_at,iniciada_at,enviada_at,activated_at,cancelled_at,dt_saida,tipo_contrato,product_nm,...,lead_elegivel,proposta_iniciada,proposta_enviada,proposta_aprovada,proposta_ativada,is_activeted,first_comunicacao_date,first_competencia_date,rating_cl_pol_blend4,rating_pol_blend4
14,4901792,2026-11-04,2026-11-04,NaT,NaT,NaT,2026-11-04,NaT,PF,None,...,False,False,False,False,False,False,NaT,NaT,None,N/I
35,4895883,2026-11-04,2026-11-04,NaT,NaT,NaT,2026-11-04,NaT,PF,None,...,False,False,False,False,False,False,NaT,NaT,None,N/I
64,4895943,2026-11-04,2026-11-04,NaT,NaT,NaT,NaT,NaT,PF,None,...,True,False,False,False,False,False,NaT,NaT,None,N/I
74,4899496,2026-11-04,2026-11-04,2026-11-04,NaT,NaT,2026-11-04,NaT,PF,Flex,...,True,True,False,False,False,False,NaT,NaT,None,N/I
99,4692331,2026-09-22,2026-09-22,NaT,NaT,NaT,2026-09-22,NaT,PF,None,...,False,False,False,False,False,False,NaT,NaT,None,N/I
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
565645,4491092,2026-08-10,2026-08-10,2026-08-10,2026-08-10,NaT,NaT,NaT,PF,Smart Plus,...,True,True,True,True,False,False,NaT,NaT,None,N/I
565655,4489406,2026-08-10,2026-08-10,2026-08-10,NaT,NaT,2026-09-10,NaT,PF,Fiança Especial,...,True,True,False,False,False,False,NaT,NaT,None,N/I
565677,4500582,2026-08-12,2026-08-12,NaT,NaT,NaT,2026-08-12,NaT,PF,None,...,False,False,False,False,False,False,NaT,NaT,None,N/I
565792,4515858,2026-08-14,2026-08-14,NaT,NaT,NaT,2026-08-14,NaT,PF,None,...,False,False,False,False,False,False,NaT,NaT,None,N/I


In [44]:
df_funil.groupby("qtd_proponentes", dropna=False).size()

qtd_proponentes
1       422796
2        18083
3         1708
4          214
5           23
6            2
8            1
<NA>    123413
dtype: int64

In [45]:
df_funil.columns

Index(['contract_id', 'dt_lead', 'requested_at', 'iniciada_at', 'enviada_at',
       'activated_at', 'cancelled_at', 'dt_saida', 'tipo_contrato',
       'product_nm', 'modeloBlend', 'bureau_nm', 'modelo_blend',
       'bureau_nm_ajust', 'is_fallback', 'total_rental_value_informed_nr',
       'rental_value_nr', 'agency_id', 'contract_city_nm', 'person_age',
       'qtd_proponentes', 'score_imobiliaria',
       'person_restriction_total_value', 'bvs_cust_score_nr',
       'blend_regressao_predict_nr', 'rating_score_ds', 'person_deemed_income',
       'pre_analysis_result', 'lead_elegivel', 'proposta_iniciada',
       'proposta_enviada', 'proposta_aprovada', 'proposta_ativada',
       'is_activeted', 'first_comunicacao_date', 'first_competencia_date',
       'rating_cl_pol_blend4', 'rating_pol_blend4'],
      dtype='object')

In [46]:
df_funil[df_funil["qtd_proponentes"].isna()].groupby("modeloBlend", dropna=False).size()

modeloBlend
BLEND3_3                36208
BLEND3_4                26767
BLEND_4                 59844
BLEND_REGRESSAO_2026      389
BVS_CUSTOM_V2              57
HFT1                       26
NaN                       122
dtype: int64